<a href="https://colab.research.google.com/github/Tomca765/Tools-for-the-Analysis-of-Histological-Images-of-Liver-Scaffolds-in-the-QuPath-Environment/blob/main/qupath_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================================
# Bubble
# =========================================================================




!pip install -q flask opencv-python-headless pillow pyngrok

import os
import base64
import threading
import cv2
import numpy as np
from flask import Flask, request, jsonify
from pyngrok import ngrok

# =========================================================================
# CONFIGURATION
# =========================================================================
NGROK_AUTH_TOKEN = "3KdNGUA1jz7IBxOfwhLcWXT7moa_36tNzYBAc1rpuoqap4dAX"
NGROK_DOMAIN     = "upstage-justice-oblong.ngrok-free.dev"

# Zastavení o bílé pozadí mimo tkáň
TISSUE_MAX_BRIGHTNESS = 245
MIN_TISSUE_SATURATION = 20

# Parametry nafukování bubliny a zastavení o změnu barvy (CIE Lab)
MAX_COLOR_DIFF    = 22.0   # Max. odchylka barvy oproti tkáni kolem žíly (nižší = zastaví se o jemnější změnu barvy)
MAX_EDGE_STRENGTH = 18.0   # Citlivost na ostré barevné přechody / septa (nižší = přísnější stěna)
SMOOTH_BLUR_SIZE  = 9      # Vyhlazení jemných struktur uvnitř lalůčku

app = Flask(__name__)

@app.route('/analyze', methods=['POST'])
def analyze_image():
    data = request.get_json(silent=True)
    if not data or 'image' not in data or 'seeds' not in data:
        return jsonify({'status': 'error', 'message': 'Missing image or seeds'}), 400

    img_bytes = base64.b64decode(data['image'])
    nparr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    if img is None:
        return jsonify({'status': 'error', 'message': 'Decode failed'}), 400

    h, w = img.shape[:2]
    seeds = data['seeds']
    seed_names = data.get('seed_names', [])
    target_index = data.get('target_index', None)  # None = Fáze 1 (vše), int = Fáze 2 (detail jedné žíly)
    num_seeds = len(seeds)

    if num_seeds == 0:
        return jsonify({
            'status': 'success',
            'contour_count': 0,
            'contours': [],
            'pink_percentages': [],
            'seed_names': []
        })

    is_fine_pass = target_index is not None
    poly_epsilon = 0.0004 if is_fine_pass else 0.0012
    ring_width = 16 if is_fine_pass else 8

    # 1. VYHLAZENÍ A PŘEVOD DO CIE LAB
    blur_k = SMOOTH_BLUR_SIZE if SMOOTH_BLUR_SIZE % 2 == 1 else SMOOTH_BLUR_SIZE + 1
    blurred_bgr = cv2.GaussianBlur(img, (blur_k, blur_k), 0)
    lab = cv2.cvtColor(blurred_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)

    gray = cv2.cvtColor(blurred_bgr, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(blurred_bgr, cv2.COLOR_BGR2HSV)
    saturation = hsv[:, :, 1]

    # Maska tkáně (zastaví bubliny o bílé pozadí kolem řezu)
    tissue_mask = ((gray < TISSUE_MAX_BRIGHTNESS) | (saturation > MIN_TISSUE_SATURATION)).astype(np.uint8) * 255
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    tissue_mask = cv2.morphologyEx(tissue_mask, cv2.MORPH_CLOSE, kernel_close)

    # Výpočet barevného gradientu (ostré změny barvy = bariéra)
    grad_sq = np.zeros((h, w), dtype=np.float32)
    for c in range(3):
        gx = cv2.Sobel(lab[:, :, c], cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(lab[:, :, c], cv2.CV_32F, 0, 1, ksize=3)
        grad_sq += (gx * gx + gy * gy) / 16.0
    edge_mag = np.sqrt(grad_sq)
    edge_ok = edge_mag <= MAX_EDGE_STRENGTH

    # 2. PŘÍPRAVA KAŽDÉ CENTRÁLNÍ ŽÍLY (SEED) A JEJÍ BAREVNÉ ZÓNY
    seed_masks = []
    allowed_masks = []
    union_allowed = np.zeros((h, w), dtype=bool)
    ring_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ring_width * 2 + 1, ring_width * 2 + 1))

    for pts in seeds:
        s_mask = np.zeros((h, w), dtype=np.uint8)
        poly = np.array(pts, dtype=np.int32).reshape((-1, 1, 2))
        cv2.fillPoly(s_mask, [poly], 255)
        seed_bool = s_mask > 0
        seed_masks.append(seed_bool)

        if not np.any(seed_bool):
            allowed_masks.append(np.zeros((h, w), dtype=bool))
            continue

        # Odečet referenční barvy tkáně těsně kolem zakroužkované žíly
        dilated = cv2.dilate(s_mask, ring_kernel)
        ring_bool = (dilated > 0) & (~seed_bool) & (tissue_mask > 0)

        # Fallback: pokud byla žíla zakroužkovaná hluboko uvnitř velkého bílého lumen, najdeme nejbližší tkáň
        if not np.any(ring_bool) and np.any(tissue_mask > 0):
            dist_from_seed = cv2.distanceTransform((~seed_bool).astype(np.uint8), cv2.DIST_L2, 5)
            tissue_dists = dist_from_seed[tissue_mask > 0]
            min_d = np.min(tissue_dists)
            ring_bool = (tissue_mask > 0) & (dist_from_seed <= min_d + ring_width)

        if np.any(ring_bool):
            ref_lab = np.median(lab[ring_bool], axis=0)
        else:
            ref_lab = np.median(lab[seed_bool], axis=0)

        # Barevná vzdálenost každého pixelu od tkáně u této žíly
        diff = lab - ref_lab
        color_dist = np.sqrt(np.sum(diff * diff, axis=2))

        raw_allowed = ((tissue_mask > 0) & (color_dist <= MAX_COLOR_DIFF) & edge_ok) | seed_bool

        # Ponechat jen souvislou oblast spojenou s touto žilou
        _, cc_labels = cv2.connectedComponents(raw_allowed.astype(np.uint8), connectivity=4)
        touching = np.unique(cc_labels[seed_bool])
        touching = touching[touching != 0]

        if len(touching) > 0:
            reachable = np.isin(cc_labels, touching)
        else:
            reachable = seed_bool

        allowed_masks.append(reachable)
        union_allowed |= reachable

    # 3. SOUČASNÉ NAFUKOVÁNÍ BUBLIN (MARKER-CONTROLLED WATERSHED)
    barrier_id = num_seeds + 1
    markers = np.full((h, w), barrier_id, dtype=np.int32)
    markers[union_allowed] = 0

    for idx, seed_bool in enumerate(seed_masks):
        markers[seed_bool] = idx + 1

    grad_norm = np.clip((edge_mag / (MAX_EDGE_STRENGTH + 1e-5)) * 255.0, 0, 255).astype(np.uint8)
    grad_bgr = cv2.merge([grad_norm, grad_norm, grad_norm])
    cv2.watershed(grad_bgr, markers)

    # 4. VYTVOŘENÍ OBDRYSŮ A VÝPOČET PINK %
    hsv_orig = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    pink1 = cv2.inRange(hsv_orig, np.array([135, 30, 50], dtype=np.uint8), np.array([175, 255, 255], dtype=np.uint8))
    pink2 = cv2.inRange(hsv_orig, np.array([0, 30, 50], dtype=np.uint8), np.array([10, 255, 255], dtype=np.uint8))
    pink_mask = cv2.bitwise_or(pink1, pink2)

    polygons = []
    pink_percentages = []
    out_names = []
    clean_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    indices_to_extract = [int(target_index)] if is_fine_pass else list(range(num_seeds))

    for idx in indices_to_extract:
        if idx < 0 or idx >= num_seeds:
            continue

        vein_id = idx + 1
        region = ((markers == vein_id) & allowed_masks[idx]) | seed_masks[idx]
        region_u8 = cv2.morphologyEx(region.astype(np.uint8) * 255, cv2.MORPH_CLOSE, clean_kernel)

        contours, _ = cv2.findContours(region_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue

        cnt = max(contours, key=cv2.contourArea)
        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue

        approx = cv2.approxPolyDP(cnt, poly_epsilon * perimeter, True)
        pts = [[int(pt[0][0]), int(pt[0][1])] for pt in approx]

        if len(pts) >= 3:
            x, y, bw, bh = cv2.boundingRect(cnt)
            cell_mask_roi = np.zeros((bh, bw), dtype=np.uint8)
            cnt_offset = (cnt - [x, y]).astype(np.int32)
            cv2.drawContours(cell_mask_roi, [cnt_offset], -1, 255, thickness=cv2.FILLED)

            total_pixels = cv2.countNonZero(cell_mask_roi)
            if total_pixels > 0:
                roi_pink = pink_mask[y:y+bh, x:x+bw]
                pink_pixels = cv2.countNonZero(cv2.bitwise_and(roi_pink, roi_pink, mask=cell_mask_roi))
                pink_pct = round((float(pink_pixels) / float(total_pixels)) * 100.0, 2)
            else:
                pink_pct = 0.0

            polygons.append(pts)
            pink_percentages.append(pink_pct)
            name = seed_names[idx] if idx < len(seed_names) and seed_names[idx] else f"Vein {vein_id}"
            out_names.append(name)

    return jsonify({
        'status': 'success',
        'contour_count': len(polygons),
        'contours': polygons,
        'pink_percentages': pink_percentages,
        'seed_names': out_names
    })

# SERVER LAUNCH
ngrok.kill()
ngrok_config = os.path.expanduser("~/.config/ngrok/ngrok.yml")
if os.path.exists(ngrok_config):
    os.remove(ngrok_config)

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000), daemon=True).start()

tunnel = ngrok.connect(5000, url=NGROK_DOMAIN)
print(f"PERMANENT ENDPOINT ACTIVE: {tunnel.public_url}/analyze")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


PERMANENT ENDPOINT ACTIVE: https://upstage-justice-oblong.ngrok-free.dev/analyze


In [1]:
# =========================================================================
# Bubble26.11.
# =========================================================================



!pip install -q flask opencv-python-headless pillow pyngrok scikit-image

import os
import base64
import threading
import cv2
import numpy as np
from flask import Flask, request, jsonify
from pyngrok import ngrok
from skimage.morphology import skeletonize
from skimage.filters import sato

# =========================================================================
# CONFIGURATION
# =========================================================================
NGROK_AUTH_TOKEN = "3KdNGUA1jz7IBxOfwhLcWXT7moa_36tNzYBAc1rpuoqap4dAX"
NGROK_DOMAIN     = "upstage-justice-oblong.ngrok-free.dev"

# Zastavení o bílé pozadí mimo tkáň
TISSUE_MAX_BRIGHTNESS = 245
MIN_TISSUE_SATURATION = 20

# Parametry nafukování bubliny a zastavení o změnu barvy (CIE Lab)
MAX_COLOR_DIFF    = 22.0   # Max. odchylka barvy oproti tkáni kolem žíly
MAX_EDGE_STRENGTH = 18.0   # Citlivost na ostré barevné přechody / septa
SMOOTH_BLUR_SIZE  = 9      # Vyhlazení jemných struktur uvnitř lalůčku

# Parametry detekce i velmi světlých čar/žil (skoro splývajících s pozadím)
LINE_DARKNESS_DROP  = 6.0   # Jasně zřetelné tmavší čáry (rozdíl jasu oproti čistému pozadí)
FAINT_DARKNESS_DROP = 1.8   # Minimální ztmavení pro velmi světlé čáry (nižší = vezme ještě světlejší)
SATO_STRONG_THRESH  = 0.045 # Silná odezva liniového filtru
SATO_WEAK_THRESH    = 0.020 # Citlivost na velmi slabé linie (nižší = zachytí slabší vlákna)
MIN_FAINT_LINE_PX   = 14    # Min. plocha (px) samostatného světlého vlákna proti tečkovanému šumu
MIN_SKELETON_PX     = 7     # Odstraní izolované útržky skeletonu kratší než 7 px

app = Flask(__name__)


def find_exact_vein_mask(lab, gray, saturation, pts):
    """
    Rychle najde přesnou centrální žílu (lumen + tenkou stěnu) uvnitř hrubého
    uživatelského zakroužkování, aby se neodečítalo její okolí.
    """
    h, w = gray.shape
    seed_mask = np.zeros((h, w), dtype=np.uint8)
    if len(pts) < 3:
        return seed_mask > 0

    poly = np.array(pts, dtype=np.int32).reshape((-1, 1, 2))
    cv2.fillPoly(seed_mask, [poly], 255)
    seed_area = cv2.countNonZero(seed_mask)
    if seed_area == 0:
        return seed_mask > 0

    sx, sy, sw, sh = cv2.boundingRect(poly)
    equiv_r = max(4.0, np.sqrt(seed_area / np.pi))
    pad = max(12, int(round(equiv_r * 0.45)))

    x0 = max(0, sx - pad)
    y0 = max(0, sy - pad)
    x1 = min(w, sx + sw + pad)
    y1 = min(h, sy + sh + pad)

    seed_c = seed_mask[y0:y1, x0:x1]
    gray_c = gray[y0:y1, x0:x1]
    sat_c  = saturation[y0:y1, x0:x1]
    lab_c  = lab[y0:y1, x0:x1]
    ch, cw = seed_c.shape

    expand_r = max(5, int(round(equiv_r * 0.35)))
    expand_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (expand_r * 2 + 1, expand_r * 2 + 1))
    search_zone = cv2.dilate(seed_c, expand_k) > 0

    # 1. Hledání světlého lumen žíly uvnitř zakroužkování
    lumen_raw = ((gray_c >= (TISSUE_MAX_BRIGHTNESS - 12)) & (sat_c <= (MIN_TISSUE_SATURATION + 15)) & search_zone).astype(np.uint8) * 255
    close_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    lumen_closed = cv2.morphologyEx(lumen_raw, cv2.MORPH_CLOSE, close_k)

    num_cc, cc_labels, stats, _ = cv2.connectedComponentsWithStats(lumen_closed, connectivity=8)
    best_id = -1
    best_overlap = 0

    for cc_id in range(1, num_cc):
        area = stats[cc_id, cv2.CC_STAT_AREA]
        if area < 12:
            continue
        bx, by, bw, bh, _ = stats[cc_id]
        touches = int(bx == 0) + int(by == 0) + int(bx + bw >= cw) + int(by + bh >= ch)
        if touches >= 2:
            continue

        comp_mask = (cc_labels == cc_id)
        overlap = int(np.sum(comp_mask & (seed_c > 0)))
        if overlap > best_overlap:
            best_overlap = overlap
            best_id = cc_id

    if best_id > 0:
        exact_c = (cc_labels == best_id).astype(np.uint8) * 255
        wall_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        exact_c = cv2.dilate(exact_c, wall_k)
        out_mask = np.zeros((h, w), dtype=bool)
        out_mask[y0:y1, x0:x1] = exact_c > 0
        return out_mask

    # 2. Alternativa: žíla bez bílého lumen (vyplněná tmavším barvivem uprostřed kruhu)
    erode_r = max(3, int(round(equiv_r * 0.25)))
    erode_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (erode_r * 2 + 1, erode_r * 2 + 1))
    inner_c = cv2.erode(seed_c, erode_k) > 0
    ring_c  = (seed_c > 0) & (~inner_c)

    if np.any(inner_c) and np.any(ring_c):
        ref_ring = np.median(lab_c[ring_c], axis=0)
        diff = np.linalg.norm(lab_c - ref_ring, axis=2)
        blob_u8 = ((seed_c > 0) & (diff >= 18.0)).astype(np.uint8) * 255
        blob_u8 = cv2.morphologyEx(blob_u8, cv2.MORPH_CLOSE, close_k)

        num_b, b_labels, b_stats, _ = cv2.connectedComponentsWithStats(blob_u8, connectivity=8)
        best_b = -1
        best_inner_ov = 0
        for b_id in range(1, num_b):
            b_area = b_stats[b_id, cv2.CC_STAT_AREA]
            if 15 <= b_area <= seed_area * 0.85:
                ov = int(np.sum((b_labels == b_id) & inner_c))
                if ov > best_inner_ov:
                    best_inner_ov = ov
                    best_b = b_id

        if best_b > 0:
            exact_c = (b_labels == best_b).astype(np.uint8) * 255
            out_mask = np.zeros((h, w), dtype=bool)
            out_mask[y0:y1, x0:x1] = exact_c > 0
            return out_mask

    # 3. Fallback: uživatel zakroužkoval žílu už přímo na těsno
    return seed_mask > 0


def mask_to_polygon(mask_bool, epsilon_ratio=0.002):
    """Převede binární masku žíly na uzavřený polygon [[x, y], ...]."""
    u8 = mask_bool.astype(np.uint8) * 255
    contours, _ = cv2.findContours(u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []
    cnt = max(contours, key=cv2.contourArea)
    perim = cv2.arcLength(cnt, True)
    if perim == 0:
        return []
    approx = cv2.approxPolyDP(cnt, epsilon_ratio * perim, True)
    return [[int(pt[0][0]), int(pt[0][1])] for pt in approx]


def trace_skeleton_polylines(skel_u8):
    """Převede 1px binární skeleton na čisté otevřené lomené čáry (polylines)."""
    h, w = skel_u8.shape
    ys, xs = np.where(skel_u8 > 0)
    if len(xs) == 0:
        return []

    neighbors = {}
    for x, y in zip(xs.tolist(), ys.tolist()):
        nbs = []
        for dy in (-1, 0, 1):
            ny = y + dy
            if ny < 0 or ny >= h:
                continue
            for dx in (-1, 0, 1):
                if dx == 0 and dy == 0:
                    continue
                nx = x + dx
                if 0 <= nx < w and skel_u8[ny, nx] > 0:
                    nbs.append((nx, ny))
        neighbors[(x, y)] = nbs

    visited_edges = set()
    polylines = []

    def edge_key(p1, p2):
        return (p1, p2) if p1 < p2 else (p2, p1)

    nodes = [pt for pt, nbs in neighbors.items() if len(nbs) != 2]
    for start in nodes:
        for nxt in neighbors[start]:
            ek = edge_key(start, nxt)
            if ek in visited_edges:
                continue
            visited_edges.add(ek)
            path = [[start[0], start[1]], [nxt[0], nxt[1]]]
            prev, curr = start, nxt
            while len(neighbors[curr]) == 2:
                n0, n1 = neighbors[curr]
                cand = n0 if n1 == prev else n1
                ek2 = edge_key(curr, cand)
                if ek2 in visited_edges:
                    break
                visited_edges.add(ek2)
                path.append([cand[0], cand[1]])
                prev, curr = curr, cand
            if len(path) >= 2:
                polylines.append(path)

    for start, nbs in neighbors.items():
        for nxt in nbs:
            ek = edge_key(start, nxt)
            if ek in visited_edges:
                continue
            visited_edges.add(ek)
            path = [[start[0], start[1]], [nxt[0], nxt[1]]]
            prev, curr = start, nxt
            while len(neighbors[curr]) == 2:
                n0, n1 = neighbors[curr]
                cand = n0 if n1 == prev else n1
                ek2 = edge_key(curr, cand)
                if ek2 in visited_edges:
                    break
                visited_edges.add(ek2)
                path.append([cand[0], cand[1]])
                prev, curr = curr, cand
            if len(path) >= 2:
                polylines.append(path)

    return polylines


@app.route('/analyze', methods=['POST'])
def analyze_image():
    data = request.get_json(silent=True)
    if not data or 'image' not in data or 'seeds' not in data:
        return jsonify({'status': 'error', 'message': 'Missing image or seeds'}), 400

    img_bytes = base64.b64decode(data['image'])
    nparr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    if img is None:
        return jsonify({'status': 'error', 'message': 'Decode failed'}), 400

    h, w = img.shape[:2]
    seeds = data['seeds']
    seed_names = data.get('seed_names', [])
    target_index = data.get('target_index', None)
    num_seeds = len(seeds)

    if num_seeds == 0:
        return jsonify({
            'status': 'success',
            'contour_count': 0,
            'contours': [],
            'vein_contours': [],
            'pink_percentages': [],
            'seed_names': []
        })

    is_fine_pass = target_index is not None
    poly_epsilon = 0.0004 if is_fine_pass else 0.0012
    ring_width = 18 if is_fine_pass else 10
    bridge_r = 5 if is_fine_pass else 3

    # 1. VYHLAZENÍ A PŘEVOD DO CIE LAB
    blur_k = SMOOTH_BLUR_SIZE if SMOOTH_BLUR_SIZE % 2 == 1 else SMOOTH_BLUR_SIZE + 1
    blurred_bgr = cv2.GaussianBlur(img, (blur_k, blur_k), 0)
    lab = cv2.cvtColor(blurred_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)

    gray_orig = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sat_orig  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[:, :, 1]

    gray = cv2.cvtColor(blurred_bgr, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(blurred_bgr, cv2.COLOR_BGR2HSV)
    saturation = hsv[:, :, 1]

    tissue_mask = ((gray < TISSUE_MAX_BRIGHTNESS) | (saturation > MIN_TISSUE_SATURATION)).astype(np.uint8) * 255
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    tissue_mask = cv2.morphologyEx(tissue_mask, cv2.MORPH_CLOSE, kernel_close)

    grad_sq = np.zeros((h, w), dtype=np.float32)
    for c in range(3):
        gx = cv2.Sobel(lab[:, :, c], cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(lab[:, :, c], cv2.CV_32F, 0, 1, ksize=3)
        grad_sq += (gx * gx + gy * gy) / 16.0
    edge_mag = np.sqrt(grad_sq)
    edge_ok = edge_mag <= MAX_EDGE_STRENGTH

    # 2. PŘÍPRAVA KAŽDÉ CENTRÁLNÍ ŽÍLY A PŘEKROČENÍ JEJÍ STĚNY DO LALŮČKU
    exact_vein_masks = []
    start_masks = []
    allowed_masks = []
    union_allowed = np.zeros((h, w), dtype=bool)
    ring_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ring_width * 2 + 1, ring_width * 2 + 1))
    bridge_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (bridge_r * 2 + 1, bridge_r * 2 + 1))

    for pts in seeds:
        seed_bool = find_exact_vein_mask(lab, gray_orig, sat_orig, pts)
        exact_vein_masks.append(seed_bool)

        if not np.any(seed_bool):
            start_masks.append(np.zeros((h, w), dtype=bool))
            allowed_masks.append(np.zeros((h, w), dtype=bool))
            continue

        # Krok o 3-5 px vně přes stěnu žíly, aby ostrý okraj stěny neuvěznil růst bubliny
        s_mask = seed_bool.astype(np.uint8) * 255
        seed_start = ((cv2.dilate(s_mask, bridge_kernel) > 0) & (tissue_mask > 0)) | seed_bool
        start_masks.append(seed_start)

        # Vzorkování referenční barvy tkáně až vně stěny žíly
        start_u8 = seed_start.astype(np.uint8) * 255
        dilated = cv2.dilate(start_u8, ring_kernel)
        ring_bool = (dilated > 0) & (~seed_start) & (tissue_mask > 0)

        if not np.any(ring_bool) and np.any(tissue_mask > 0):
            dist_from_seed = cv2.distanceTransform((~seed_start).astype(np.uint8), cv2.DIST_L2, 5)
            tissue_dists = dist_from_seed[tissue_mask > 0]
            min_d = np.min(tissue_dists)
            ring_bool = (tissue_mask > 0) & (dist_from_seed <= min_d + ring_width)

        if np.any(ring_bool):
            ref_lab = np.median(lab[ring_bool], axis=0)
        else:
            ref_lab = np.median(lab[seed_start], axis=0)

        diff = lab - ref_lab
        color_dist = np.sqrt(np.sum(diff * diff, axis=2))

        raw_allowed = ((tissue_mask > 0) & (color_dist <= MAX_COLOR_DIFF) & edge_ok) | seed_start

        _, cc_labels = cv2.connectedComponents(raw_allowed.astype(np.uint8), connectivity=4)
        touching = np.unique(cc_labels[seed_start])
        touching = touching[touching != 0]

        if len(touching) > 0:
            reachable = np.isin(cc_labels, touching)
        else:
            reachable = seed_start

        allowed_masks.append(reachable)
        union_allowed |= reachable

    # 3. SOUČASNÉ NAFUKOVÁNÍ BUBLIN (MARKER-CONTROLLED WATERSHED)
    barrier_id = num_seeds + 1
    markers = np.full((h, w), barrier_id, dtype=np.int32)
    markers[union_allowed] = 0

    for idx, s_start in enumerate(start_masks):
        markers[s_start] = idx + 1

    grad_norm = np.clip((edge_mag / (MAX_EDGE_STRENGTH + 1e-5)) * 255.0, 0, 255).astype(np.uint8)
    grad_bgr = cv2.merge([grad_norm, grad_norm, grad_norm])
    cv2.watershed(grad_bgr, markers)

    # 4. VYTVOŘENÍ OBRYSŮ BUBLIN, PŘESNÝCH ŽIL A VÝPOČET PINK %
    hsv_orig = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    pink1 = cv2.inRange(hsv_orig, np.array([135, 30, 50], dtype=np.uint8), np.array([175, 255, 255], dtype=np.uint8))
    pink2 = cv2.inRange(hsv_orig, np.array([0, 30, 50], dtype=np.uint8), np.array([10, 255, 255], dtype=np.uint8))
    pink_mask = cv2.bitwise_or(pink1, pink2)

    polygons = []
    vein_polygons = []
    pink_percentages = []
    out_names = []
    clean_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    indices_to_extract = [int(target_index)] if is_fine_pass else list(range(num_seeds))

    for idx in indices_to_extract:
        if idx < 0 or idx >= num_seeds:
            continue

        vein_id = idx + 1
        region = ((markers == vein_id) & allowed_masks[idx]) | start_masks[idx]
        region_u8 = cv2.morphologyEx(region.astype(np.uint8) * 255, cv2.MORPH_CLOSE, clean_kernel)

        contours, _ = cv2.findContours(region_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue

        cnt = max(contours, key=cv2.contourArea)
        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue

        approx = cv2.approxPolyDP(cnt, poly_epsilon * perimeter, True)
        pts = [[int(pt[0][0]), int(pt[0][1])] for pt in approx]

        if len(pts) >= 3:
            x, y, bw, bh = cv2.boundingRect(cnt)
            cell_mask_roi = np.zeros((bh, bw), dtype=np.uint8)
            cnt_offset = (cnt - [x, y]).astype(np.int32)
            cv2.drawContours(cell_mask_roi, [cnt_offset], -1, 255, thickness=cv2.FILLED)

            total_pixels = cv2.countNonZero(cell_mask_roi)
            if total_pixels > 0:
                roi_pink = pink_mask[y:y+bh, x:x+bw]
                pink_pixels = cv2.countNonZero(cv2.bitwise_and(roi_pink, roi_pink, mask=cell_mask_roi))
                pink_pct = round((float(pink_pixels) / float(total_pixels)) * 100.0, 2)
            else:
                pink_pct = 0.0

            v_poly = mask_to_polygon(exact_vein_masks[idx], epsilon_ratio=0.002)

            polygons.append(pts)
            vein_polygons.append(v_poly)
            pink_percentages.append(pink_pct)
            name = seed_names[idx] if idx < len(seed_names) and seed_names[idx] else f"Vein {vein_id}"
            out_names.append(name)

    return jsonify({
        'status': 'success',
        'contour_count': len(polygons),
        'contours': polygons,
        'vein_contours': vein_polygons,
        'pink_percentages': pink_percentages,
        'seed_names': out_names
    })


@app.route('/skeletonize', methods=['POST'])
def skeletonize_lines():
    data = request.get_json(silent=True)
    if not data or 'image' not in data or 'cell_polygon' not in data:
        return jsonify({'status': 'error', 'message': 'Missing image or cell_polygon'}), 400

    img_bytes = base64.b64decode(data['image'])
    nparr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    if img is None:
        return jsonify({'status': 'error', 'message': 'Decode failed'}), 400

    h, w = img.shape[:2]

    # Maska samotné buňky
    cell_mask = np.zeros((h, w), dtype=np.uint8)
    cell_poly = np.array(data['cell_polygon'], dtype=np.int32).reshape((-1, 1, 2))
    cv2.fillPoly(cell_mask, [cell_poly], 255)

    # Jemné odšumění a kombinace jasu L + zeleného kanálu G
    smooth = cv2.GaussianBlur(img, (3, 3), 0)
    lab = cv2.cvtColor(smooth, cv2.COLOR_BGR2LAB).astype(np.float32)
    gray_orig = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sat_orig  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[:, :, 1]

    # Odpočet POUZE přesné centrální žíly (ne celého hrubého kruhu)
    exclude_seeds = data.get('exclude_seeds', [])
    is_exact_vein = bool(data.get('is_exact_vein', False))
    wall_margin_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    for s_pts in exclude_seeds:
        if len(s_pts) >= 3:
            if is_exact_vein:
                s_mask = np.zeros((h, w), dtype=np.uint8)
                s_poly = np.array(s_pts, dtype=np.int32).reshape((-1, 1, 2))
                cv2.fillPoly(s_mask, [s_poly], 255)
                s_mask = cv2.dilate(s_mask, wall_margin_k)
                cell_mask[s_mask > 0] = 0
            else:
                exact_vein = find_exact_vein_mask(lab, gray_orig, sat_orig, s_pts)
                cell_mask[exact_vein] = 0

    L = lab[:, :, 0]
    G = smooth[:, :, 1].astype(np.float32)
    signal = 0.5 * L + 0.5 * G

    # Maska tkáně (ignorujeme čistě bílé otvory)
    gray = cv2.cvtColor(smooth, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(smooth, cv2.COLOR_BGR2HSV)
    tissue_mask = ((gray < TISSUE_MAX_BRIGHTNESS) | (hsv[:, :, 1] > MIN_TISSUE_SATURATION)) & (cell_mask > 0)

    # Mírná eroze vnějšího okraje buňky (3 px), aby se nepočítala samotná hranice buňky
    erode_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    inner_mask = (cv2.erode(cell_mask, erode_k) > 0) & tissue_mask

    if not np.any(inner_mask):
        return jsonify({
            'status': 'success',
            'length_px': 0.0,
            'area_px': float(cv2.countNonZero(cell_mask)),
            'skeleton_contours': []
        })

    # 1. Morfologický odhad skutečného světlého pozadí
    bg_morph_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17))
    bg_closed = cv2.morphologyEx(signal, cv2.MORPH_CLOSE, bg_morph_k)
    bg_smooth = cv2.GaussianBlur(bg_closed, (31, 31), 0)
    darkness_diff = bg_smooth - signal

    # 2. Kontrastní normalizace pro Sato filtr
    p95 = float(np.percentile(darkness_diff[inner_mask], 95))
    denom = max(16.0, p95)
    norm_signal = np.clip(darkness_diff, 0.0, None) / denom
    norm_signal = np.clip(norm_signal, 0.0, 1.5)

    ridge_resp = sato(norm_signal, sigmas=[1.0, 1.8, 2.8], black_ridges=False)

    # 3. Hysterezní prahování
    strong_mask = inner_mask & (
        (darkness_diff >= LINE_DARKNESS_DROP) |
        ((ridge_resp >= SATO_STRONG_THRESH) & (darkness_diff >= FAINT_DARKNESS_DROP))
    )
    weak_mask = inner_mask & (
        (ridge_resp >= SATO_WEAK_THRESH) & (darkness_diff >= FAINT_DARKNESS_DROP)
    )

    weak_u8 = weak_mask.astype(np.uint8)
    num_w, w_labels, w_stats, _ = cv2.connectedComponentsWithStats(weak_u8, connectivity=8)
    raw_lines = strong_mask.copy()

    for cc_id in range(1, num_w):
        x, y, bw, bh, area = w_stats[cc_id]
        comp_crop = (w_labels[y:y+bh, x:x+bw] == cc_id)
        strong_crop = strong_mask[y:y+bh, x:x+bw]
        if np.any(comp_crop & strong_crop) or area >= MIN_FAINT_LINE_PX:
            raw_lines[y:y+bh, x:x+bw] |= comp_crop

    close_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    lines_u8 = cv2.morphologyEx(raw_lines.astype(np.uint8) * 255, cv2.MORPH_CLOSE, close_k)

    num_cc, cc_labels, stats, _ = cv2.connectedComponentsWithStats(lines_u8, connectivity=8)
    clean_lines = np.zeros_like(lines_u8)
    for cc_id in range(1, num_cc):
        if stats[cc_id, cv2.CC_STAT_AREA] >= MIN_SKELETON_PX:
            clean_lines[cc_labels == cc_id] = 255

    skel_bool = skeletonize(clean_lines > 0)
    skel_u8 = skel_bool.astype(np.uint8)

    num_sk, sk_labels, sk_stats, _ = cv2.connectedComponentsWithStats(skel_u8, connectivity=8)
    filtered_skel = np.zeros_like(skel_u8)
    for sk_id in range(1, num_sk):
        if sk_stats[sk_id, cv2.CC_STAT_AREA] >= MIN_SKELETON_PX:
            filtered_skel[sk_labels == sk_id] = 1
    skel_u8 = filtered_skel

    horiz = np.sum(skel_u8[:, :-1] & skel_u8[:, 1:])
    vert  = np.sum(skel_u8[:-1, :] & skel_u8[1:, :])
    diag1 = np.sum(skel_u8[:-1, :-1] & skel_u8[1:, 1:])
    diag2 = np.sum(skel_u8[:-1, 1:] & skel_u8[1:, :-1])

    l_corners = (
        np.sum(skel_u8[:-1, :-1] & skel_u8[:-1, 1:] & skel_u8[1:, :-1]) +
        np.sum(skel_u8[:-1, :-1] & skel_u8[:-1, 1:] & skel_u8[1:, 1:]) +
        np.sum(skel_u8[1:, :-1] & skel_u8[1:, 1:] & skel_u8[:-1, :-1]) +
        np.sum(skel_u8[1:, :-1] & skel_u8[1:, 1:] & skel_u8[:-1, 1:])
    )
    total_length_px = float(horiz + vert) + float(diag1 + diag2) * np.sqrt(2.0) - float(l_corners)
    total_length_px = max(0.0, total_length_px)

    out_polylines = trace_skeleton_polylines(skel_u8)

    return jsonify({
        'status': 'success',
        'length_px': round(total_length_px, 2),
        'area_px': float(cv2.countNonZero(cell_mask)),
        'skeleton_contours': out_polylines
    })


# SERVER LAUNCH
ngrok.kill()
ngrok_config = os.path.expanduser("~/.config/ngrok/ngrok.yml")
if os.path.exists(ngrok_config):
    os.remove(ngrok_config)

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000), daemon=True).start()

tunnel = ngrok.connect(5000, url=NGROK_DOMAIN)
print(f"PERMANENT ENDPOINT ACTIVE: {tunnel.public_url}/analyze & /skeletonize")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


PERMANENT ENDPOINT ACTIVE: https://upstage-justice-oblong.ngrok-free.dev/analyze & /skeletonize
